In [ ]:
import httpx
import pendulum
import polars as pl
import scipy.stats
from hishel.httpx import SyncCacheClient
from liblaf import grapes
from pendulum import DateTime

from qoc.market import MarketDataFuturesUsds

In [ ]:
METRIC_NAMES: list[str] = [
    "daa",
    "fdv",
    "fees",
    "market_cap",
    "profit",
    "rent_paid",
    "stables_mcap",
    "throughput",
    "tvl",
    "txcosts",
    "txcount",
]
ORIGIN_KEY_TO_SYMBOL: dict[str, str] = {
    "arbitrum": "ARBUSDT",
    "base": "ETHUSDT",
    "celo": "CELOUSDT",
    "ethereum": "ETHUSDT",
    "fraxtal": "FXSUSDT",
    "gravity": "GUSDT",
    "imx": "IMXUSDT",
    "lisk": "LSKUSDT",
    "manta": "MANTAUSDT",
    "metis": "METISUSDT",
    "optimism": "OPUSDT",
    "plume": "PLUMEUSDT",
    "polygon_zkevm": "POLUSDT",
    "scroll": "SCRUSDT",
    "starknet": "STRKUSDT",
    "taiko": "TAIKOUSDT",
    "unichain": "UNIUSDT",
    "worldchain": "WLDUSDT",
    "zksync_era": "ZKUSDT",
}
start: DateTime = pendulum.datetime(2023, 1, 1)
end: DateTime = pendulum.datetime(2024, 1, 1)

In [ ]:
client = SyncCacheClient()
data: pl.DataFrame = pl.DataFrame()
metric_names: list[str] = []
for metric_key in METRIC_NAMES:
    response: httpx.Response = client.get(
        f"https://api.growthepie.com/v1/export/{metric_key}.json",
    )
    metric: pl.DataFrame = pl.from_records(response.json()).cast({"date": pl.Date()})
    metric = metric.sort(pl.col("date")).filter(
        pl.col("date").is_between(start.date(), end.date())
    )
    metric_names += metric["metric_key"].unique().to_list()
    metric = metric.with_columns(
        pl.concat_str(pl.col("origin_key"), pl.col("metric_key"), separator="_").alias(
            "origin_metric"
        )
    ).pivot("origin_metric", index="date", values="value")
    data = (
        metric
        if data.is_empty()
        else data.join(metric, on="date", how="full", validate="1:1", coalesce=True)
    )

In [ ]:
market_data = MarketDataFuturesUsds()
for origin_key, symbol in ORIGIN_KEY_TO_SYMBOL.items():
    klines: pl.DataFrame = market_data.klines(symbol, "1d", start=start, end=end)
    klines = klines.select(
        pl.col("close").alias(f"{origin_key}_price"),
        pl.col("open_time").dt.date().alias("date"),
    )
    data = data.join(klines, on="date", how="full", validate="1:1", coalesce=True)

In [ ]:
origin_key: str = "imx"
metric_key: str = "daa"
price_shift: int = 30
metric_shift: int = 30

price_key: str = f"{origin_key}_price"
origin_metric_key: str = f"{origin_key}_{metric_key}"
valid_data: pl.DataFrame = data.select(
    pl.col("date"),
    (
        (pl.col(price_key).shift(-price_shift - 1) - pl.col(price_key).shift(-1))
        / pl.col(price_key).shift(-1)
    ).alias(f"{price_key}_{price_shift}d_change"),
    (
        (
            pl.col(origin_metric_key).shift(1)
            - pl.col(origin_metric_key).shift(metric_shift + 1)
        )
        / pl.col(origin_metric_key).shift(metric_shift + 1)
    ).alias(f"{origin_metric_key}_{metric_shift}d_change"),
).drop_nulls()
valid_data